# PMLS - ASSIGNMENT 2

## Developing API For Excel

### Business Context:
A skin clinic conducted a marketing campaign targeting 10,000 customers. The objective is to analyze which customer segments are more likely to respond to the campaign based on demographics and purchase behavior. Insights from this analysis will help improve future targeting strategies and increase campaign effectiveness.

### Data Dictionary:
|        Column Name        |                        Description                         |
|:-------------------------:|:----------------------------------------------------------:|
|          CustID           |            Unique identifier for each customer             |
|          Gender           |             Gender of customer (Male, Female)              |
|         AgeGroup          |         Age bracket of customer (<30, 30-50, >50)          |
|   Purchase_Last_Quarter   | Whether customer made a purchase in last quarter (Yes/No)  |
| Unique_Products_Purchased |    Number of unique products purchased in the last year    |
|   Response_to_Campaign    |       Target variable: Response to campaign (Yes/No)       |


In [1]:
import pandas as pd
import numpy as np

### 1. Data Import and Exploratory Data Analysis (EDA)

In [2]:
data = pd.read_csv("skin clinic campaign.csv")
data.head()


,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes


In [3]:
data.shape

(10000, 6)

In [4]:
data.dtypes

CustID                        int64
Gender                       object
AgeGroup                     object
Purchase_Last_Quarter        object
Unique_Products_Purchased     int64
Response_to_Campaign         object
dtype: object

In [5]:
data.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 6 columns):
 #   Column                     Non-Null Count  Dtype 
---  ------                     --------------  ----- 
 0   CustID                     10000 non-null  int64 
 1   Gender                     10000 non-null  object
 2   AgeGroup                   10000 non-null  object
 3   Purchase_Last_Quarter      10000 non-null  object
 4   Unique_Products_Purchased  10000 non-null  int64 
 5   Response_to_Campaign       10000 non-null  object
dtypes: int64(2), object(4)
memory usage: 468.9+ KB


In [6]:
data.nunique()

CustID                       10000
Gender                           2
AgeGroup                         3
Purchase_Last_Quarter            2
Unique_Products_Purchased       15
Response_to_Campaign             2
dtype: int64

In [7]:
data.isnull().sum()

CustID                       0
Gender                       0
AgeGroup                     0
Purchase_Last_Quarter        0
Unique_Products_Purchased    0
Response_to_Campaign         0
dtype: int64

In [8]:
data_copy = data.copy()

data_copy

,CustID,Gender,AgeGroup,Purchase_Last_Quarter,Unique_Products_Purchased,Response_to_Campaign
0,100001,Female,30-50,Yes,9,Yes
1,100002,Male,>50,Yes,15,No
2,100003,Male,>50,Yes,11,No
3,100004,Female,30-50,No,6,No
4,100005,Female,<30,Yes,13,Yes
...,...,...,...,...,...,...
9995,109996,Male,>50,Yes,13,Yes
9996,109997,Female,<30,No,5,No
9997,109998,Female,30-50,Yes,15,Yes
9998,109999,Male,30-50,No,2,No


### 2. Response Rate Calculation Engine

We define a reusable function to compute segment-level statistics:
- **Total Customers**: Total count of individuals in the segment
- **Responders**: Customers who responded with 'Yes'
- **Non-Responders**: Customers who did not respond ('No')
- **Response Rate (%)**: Percentage of responders out of total customers in the segment


In [9]:
def calculate_response_rate(data_copy, column, category_order=None):
    categories = category_order if category_order else list(data_copy[column].dropna().unique())
    results = []
    
    for category in categories:
        category_data = data_copy[data_copy[column] == category]
        total_customers = len(category_data)
        responders = int((category_data['Response_to_Campaign'] == 'Yes').sum())
        non_responders = total_customers - responders
        response_rate = (responders / total_customers * 100) if total_customers > 0 else 0
        
        results.append({
            "segment": str(category),
            "total_customers": total_customers,
            "responders": responders,
            "non_responders": non_responders,
            "response_rate_pct": round(response_rate, 2)
        })
        
    return pd.DataFrame(results)

### Task 1: Gender vs Campaign Response
Create a table showing the response rate (%) for Male and Female customers.


In [10]:
gender_table = calculate_response_rate(data_copy, "Gender", category_order= ["Female", "Male"])

gender_table

,segment,total_customers,responders,non_responders,response_rate_pct
0,Female,6007,2629,3378,43.77
1,Male,3993,1361,2632,34.08


**Interpretation** - Female customers showed a significantly higher response rate (43.77%) compared to male customers (34.08%). Females also represent the majority of the customer base (60.07%), making them the primary engagement driver for clinic promotions.

### Task 2: Age Group vs Campaign Response
Prepare a table showing the response rate (%) across the following age groups:
- `<30`
- `30-50`
- `>50`


In [11]:
age_table = calculate_response_rate(data_copy, "AgeGroup", category_order=["<30", "30-50", ">50"])
age_table


,segment,total_customers,responders,non_responders,response_rate_pct
0,<30,2499,823,1676,32.93
1,30-50,4992,2349,2643,47.06
2,>50,2509,818,1691,32.60


**Interpretation** - Customers in the 30–50 age group demonstrated the highest campaign response rate at 47.06%, whereas younger customers (<30) and older customers (>50) responded at lower, nearly identical rates of 32.93% and 32.60% respectively.


### Task 3: Purchase in Last Quarter vs Campaign Response
Construct a table showing response rate (%) for customers based on whether they made a purchase in the last quarter (Yes/No).


In [15]:
purchase_table = calculate_response_rate(
    data_copy,
    "Purchase_Last_Quarter",
    category_order= ["Yes", "No"]
)

purchase_table

,segment,total_customers,responders,non_responders,response_rate_pct
0,Yes,6514,3234,3280,49.65
1,No,3486,756,2730,21.69


**Interpretation** - Recency of purchase is a powerful predictor of response. Customers who purchased in the last quarter responded at 49.65%, more than double the response rate of dormant customers (21.69%).


### Task 4: Product Usage vs Campaign Response
Categorise customers based on the number of unique products purchased in the last year: (1–4, 5–8, >8)

Create a table showing the response rate (%) for each category.


In [16]:
data_copy["Product_Usage"] = pd.cut(
    data_copy["Unique_Products_Purchased"],
    bins= [0, 4, 8, np.inf],
    labels= ["1-4", "5-8", ">8"]
)

data_copy["Product_Usage"].value_counts()


Product_Usage
>8     4206
5-8    3703
1-4    2091
Name: count, dtype: int64

In [17]:
product_usage_table = calculate_response_rate(
    data_copy,
    "Product_Usage",
    category_order= ["1-4", "5-8", ">8"]
)

product_usage_table

,segment,total_customers,responders,non_responders,response_rate_pct
0,1-4,2091,378,1713,18.08
1,5-8,3703,1425,2278,38.48
2,>8,4206,2187,2019,52.00


**Interpretation** - Higher product variety strongly correlates with campaign receptivity. Customers purchasing >8 unique products responded at 52.00%, followed by 38.48% for 5–8 products, and only 18.08% for low-diversity purchasers (1–4 products).

### 3. Consolidated Multi-Segment Summary Table

In [18]:
consolidated_list = []

for dim_name, df_dim in [
    ("Gender", gender_table),
    ("Age Group", age_table),
    ("Purchase Last Quarter", purchase_table),
    ("Product Usage", product_usage_table)
]:
    df_temp = df_dim.copy()
    df_temp.insert(0, "dimension", dim_name)
    consolidated_list.append(df_temp)

consolidated_table = pd.concat(consolidated_list, ignore_index= True)
consolidated_table


,dimension,segment,total_customers,responders,non_responders,response_rate_pct
0,Gender,Female,6007,2629,3378,43.77
1,Gender,Male,3993,1361,2632,34.08
2,Age Group,<30,2499,823,1676,32.93
3,Age Group,30-50,4992,2349,2643,47.06
4,Age Group,>50,2509,818,1691,32.60
5,Purchase Last Quarter,Yes,6514,3234,3280,49.65
6,Purchase Last Quarter,No,3486,756,2730,21.69
7,Product Usage,1-4,2091,378,1713,18.08
8,Product Usage,5-8,3703,1425,2278,38.48
9,Product Usage,>8,4206,2187,2019,52.00


### 4. Strategic Business Insights & Recommendations
1. **Targeting Persona**: The ideal customer segment for future marketing campaigns comprises Females aged 30–50 who made a purchase in the last quarter and use >8 unique clinic products.
2. **Reactivation Strategies**: Customers with no purchase in the last quarter have a low response rate (21.69%). Rather than high-cost direct marketing, test win-back incentives or tailored re-engagement journeys.
3. **Cross-Selling & Product Breadth**: Encouraging customers to try products across multiple categories (moving from 1–4 to 5+ products) substantially boosts engagement and retention.